# 🏪 Analisis Asosiasi Keranjang Belanja (Apriori) - Toko Setia Ciawi
Notebook ini berisi dokumentasi dan implementasi **Market Basket Analysis** (Analisis Keranjang Belanja) menggunakan algoritma **Apriori** untuk menemukan pola keterkaitan pembelian produk oleh pelanggan.

### Alur Analisis:
1. **Load Data**: Membaca dataset historis transaksi `data.csv`.
2. **Pembersihan Data (Data Cleaning)**: Menyinkronkan nama barang yang typo agar data keranjang belanja akurat.
3. **Persiapan Transaksi**: Mengelompokkan item belanjaan berdasarkan ID Transaksi menjadi list keranjang belanja.
4. **One-Hot Encoding**: Mengubah list transaksi menjadi matriks biner (Boolean DataFrame) menggunakan `TransactionEncoder`.
5. **Mencari Frequent Itemsets**: Menemukan kombinasi barang yang sering dibeli bersamaan menggunakan algoritma **Apriori** (min_support = 2%).
6. **Pembentukan Aturan Asosiasi**: Membangun aturan rekomendasi produk (*Association Rules*) berdasarkan nilai confidence (min_confidence = 30%) dan lift ratio.

In [1]:
# 1. Import Library yang Dibutuhkan
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

## 1. Load Data & Pembersihan Data
Membaca file dataset `data.csv` yang berisi riwayat penjualan toko kelontong.

In [2]:
# Membaca dataset
df = pd.read_csv('data.csv')

print(f"Dataset berhasil dimuat. Total data: {df.shape[0]} baris, {df.shape[1]} kolom.")
df.head()

Dataset berhasil dimuat. Total data: 467 baris, 5 kolom.


,ID_Transaksi,Tanggal,Nama_barang,Jumlah,Satuan
0,TRX-001,2026 - 5 - 18,Telor,0.5,Kg
1,TRX-001,2026 - 5 - 18,Bawang Merah,1.0,Ons
2,TRX-001,2026 - 5 - 18,Bawang Putih,0.5,Ons
3,TRX-002,2026 - 5 - 18,Rokok Jarum Super,1.0,Bungkus
4,TRX-002,2026 - 5 - 18,Chiki Balls,2.0,Bungkus


In [3]:
# === PROSES DATA CLEANING ===
# 1. Bersihkan pemisah desimal koma menjadi titik pada kolom Jumlah
df['Jumlah'] = df['Jumlah'].astype(str).str.replace(',', '.').astype(float)

# 2. Standarisasi nama produk yang typo / duplikat
mapping_typo = {
    'Rokok Jarum Cokelat': 'Rokok Jarum Coklat',
    'Rokok Jarim Coklat': 'Rokok Jarum Coklat',
    'Jarum Super': 'Rokok Jarum Super',
    'Rokok Super': 'Rokok Jarum Super',
    'Rokok GGM': 'Rokok Garam Merah',
    'Garam': 'Rokok Garam Merah',
    'Gula Pasir Pasir': 'Gula Pasir',
}
df['Nama_barang'] = df['Nama_barang'].replace(mapping_typo)

df.loc[df['Satuan'] == 'Batang', 'Nama_barang'] = df.loc[df['Satuan'] == 'Batang', 'Nama_barang'] + ' (batang)'

print(f"Dataset berhasil dibersihkan. Total barang unik sekarang: {df['Nama_barang'].nunique()}")
df.head()

Dataset berhasil dibersihkan. Total barang unik sekarang: 93


,ID_Transaksi,Tanggal,Nama_barang,Jumlah,Satuan
0,TRX-001,2026 - 5 - 18,Telor,0.5,Kg
1,TRX-001,2026 - 5 - 18,Bawang Merah,1.0,Ons
2,TRX-001,2026 - 5 - 18,Bawang Putih,0.5,Ons
3,TRX-002,2026 - 5 - 18,Rokok Jarum Super,1.0,Bungkus
4,TRX-002,2026 - 5 - 18,Chiki Balls,2.0,Bungkus


## 2. Persiapan Transaksi (Preprocessing)
Mengelompokkan produk berdasarkan `ID_Transaksi` menjadi format list transaksi kasir.

In [4]:
print("=== Mempersiapkan Transaksi untuk Apriori ===")
transactions_df = df.groupby('ID_Transaksi')['Nama_barang'].apply(list).reset_index()
transactions = transactions_df['Nama_barang'].tolist()

print(f"Total transaksi kasir terdeteksi: {len(transactions)} keranjang.")
print("Contoh 5 keranjang transaksi pertama:")
for i, t in enumerate(transactions[:5], 1):
    print(f"  Keranjang {i}: {t}")

=== Mempersiapkan Transaksi untuk Apriori ===
Total transaksi kasir terdeteksi: 292 keranjang.
Contoh 5 keranjang transaksi pertama:
  Keranjang 1: ['Telor', 'Bawang Merah', 'Bawang Putih']
  Keranjang 2: ['Rokok Jarum Super', 'Chiki Balls']
  Keranjang 3: ['Choki Choki']
  Keranjang 4: ['Air mineral', 'Permen']
  Keranjang 5: ['Teh pucuk']


## 3. Matriks Transaksi (One-Hot Encoding)
Mengubah format list transaksi menjadi representasi tabel biner (True/False).

In [5]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(f"Dimensi matriks biner transaksi: {df_encoded.shape[0]} baris x {df_encoded.shape[1]} produk.")
df_encoded.head()

Dimensi matriks biner transaksi: 292 baris x 93 produk.


,Aida,Air Mineral,Air mineral,Amplop,Baterai,Baterai ABC,Bawang,Bawang Merah,Bawang Putih,Beng-Beng,...,Telur,Tembakau Jangkar,Tepung Serbaguna,Tepung Tapioka,Tepung Tepung Terigu,Tepung Terigu,Terasi,Tisu,Tolak Angin,Wafer Super Star
0,False,False,False,False,False,False,False,True,True,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## 4. Algoritma Apriori (Mencari Frequent Itemsets)
Mencari kombinasi itemset barang yang memenuhi nilai batas minimal support 2% (`min_support = 0.02`).

In [6]:
print("=== Menjalankan Algoritma Apriori ===")
min_support = 0.02
frequent_itemsets = apriori(df_encoded, min_support=min_support, use_colnames=True)

# Hitung ukuran kombinasi barang dan frekuensi pembelian
frequent_itemsets['Itemset Size'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
frequent_itemsets['Frequency'] = (frequent_itemsets['support'] * len(df_encoded)).round(0)

frequent_itemsets_display = frequent_itemsets[frequent_itemsets['Itemset Size'] >= 2].copy()
frequent_itemsets_display = frequent_itemsets_display[['support', 'itemsets', 'Itemset Size', 'Frequency']]
frequent_itemsets_display.columns = ['Support', 'Itemsets', 'Itemset Size', 'Frequency']

print(f"Berhasil menemukan {len(frequent_itemsets)} frequent itemsets total (tunggal + kombinasi).")
print(f"Jumlah kombinasi itemset (jumlah barang >= 2): {len(frequent_itemsets_display)}.")

frequent_itemsets_display.head(10)

=== Menjalankan Algoritma Apriori ===
Berhasil menemukan 24 frequent itemsets total (tunggal + kombinasi).
Jumlah kombinasi itemset (jumlah barang >= 2): 5.


,Support,Itemsets,Itemset Size,Frequency
19,0.023973,"frozenset({Minyak Goreng Curah, Masako})",2,7.0
20,0.020548,"frozenset({Masako, Tepung Tapioka})",2,6.0
21,0.023973,"frozenset({Minyak Goreng Curah, Tepung Tapioka})",2,7.0
22,0.030822,"frozenset({Minyak Goreng Curah, Tepung Terigu})",2,9.0
23,0.020548,"frozenset({Minyak Goreng Curah, Masako, Tepung...",3,6.0


## 5. Pembentukan Aturan Asosiasi (Association Rules)
Mencari aturan keterkaitan produk berdasarkan minimal confidence 30% (`min_confidence = 0.3`).

In [7]:
print("=== Pembentukan Aturan Asosiasi ===")
min_confidence = 0.3
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)

# Format agar mudah dibaca
rules['Rule'] = rules['antecedents'].apply(lambda x: ', '.join(list(x))) + ' -> ' + rules['consequents'].apply(lambda x: ', '.join(list(x)))
rules_display = rules.sort_values(by=['confidence', 'lift'], ascending=[False, False]).reset_index(drop=True)
rules_display = rules_display[['Rule', 'antecedent support', 'consequent support', 'support', 'confidence', 'lift']]
rules_display.columns = ['Rule', 'Antecedent Support', 'Consequent Support', 'Support', 'Confidence', 'Lift']

print(f"Berhasil menghasilkan {len(rules_display)} aturan asosiasi.")
rules_display.head(10)

=== Pembentukan Aturan Asosiasi ===
Berhasil menghasilkan 10 aturan asosiasi.


,Rule,Antecedent Support,Consequent Support,Support,Confidence,Lift
0,"Masako, Tepung Tapioka -> Minyak Goreng Curah",0.020548,0.133562,0.020548,1.000000,7.487179
1,Masako -> Minyak Goreng Curah,0.027397,0.133562,0.023973,0.875000,6.551282
2,"Minyak Goreng Curah, Tepung Tapioka -> Masako",0.023973,0.027397,0.020548,0.857143,31.285714
3,"Minyak Goreng Curah, Masako -> Tepung Tapioka",0.023973,0.034247,0.020548,0.857143,25.028571
4,"Masako -> Minyak Goreng Curah, Tepung Tapioka",0.027397,0.023973,0.020548,0.750000,31.285714
5,Masako -> Tepung Tapioka,0.027397,0.034247,0.020548,0.750000,21.900000
6,Tepung Tapioka -> Minyak Goreng Curah,0.034247,0.133562,0.023973,0.700000,5.241026
7,"Tepung Tapioka -> Minyak Goreng Curah, Masako",0.034247,0.023973,0.020548,0.600000,25.028571
8,Tepung Tapioka -> Masako,0.034247,0.027397,0.020548,0.600000,21.900000
9,Tepung Terigu -> Minyak Goreng Curah,0.065068,0.133562,0.030822,0.473684,3.546559
